# Recursive DRAG / time-dependent Schrieffer--Wolff derivation

This notebook derives the DRAG controls and the generator **order by order**.  It does not assume that the two displayed generator terms in the 2009 paper already constitute the fifth-order transformation.  Instead, after every order it extracts the remaining leakage block, solves the corresponding homological (Schrieffer--Wolff) equation, and adds the resulting $S^{(n)}$ to the *single* generator

$$S=\sum_{n=1}^{N}\epsilon^n S^{(n)},\qquad W=e^S.$$

The workflow has two stages:

1. derive the constant-$\Delta$, constant-$\lambda$ reference and require exact agreement with Eq. (10) of Motzoi *et al.*;
2. only after this regression test passes, retain $\dot\Delta$ and $\dot\lambda$ and derive the generalized controls.

Primary reference: [Motzoi *et al.*, PRL 103, 110501 (2009)](https://doi.org/10.1103/PhysRevLett.103.110501).

## 1. Convention and perturbative counting

We use the right-nested convention

$$[A,S]_0=A,\qquad [A,S]_n=[[A,S]_{n-1},S],$$

and therefore

$$H^W=\sum_{n\ge0}\frac{[H,S]_n}{n!}-i\sum_{n\ge0}\frac{[\dot S,S]_n}{(n+1)!}.$$

For the 2009 fifth-order formulas the adiabatic pulse scaling $t_g\sim1/\mathcal E_\pi$ is used. Hence one time derivative raises the perturbative order by one: $\partial_t\mapsto\epsilon\partial_t$. This is why the leading $\mathcal E^y$ term has order two and the term $\mathcal E_\pi^4\dot{\mathcal E}_\pi/\Delta^5$ has order six and is **not** part of a fifth-order result.

> This ordering must not be mixed with a convention in which derivatives retain the same order. Mixing both conventions is precisely what produces extra fifth-order $\mathcal E^y$ terms and incorrect coefficients.

In [ ]:
import sympy as sp
from IPython.display import Math, Markdown, display

sp.init_printing()
t = sp.symbols('t', real=True)
I = sp.I
eps = sp.symbols('epsilon', real=True)
MAX_ORDER = 5

## 2. Three-level operator basis and control ansatz

The physical controls are expanded with the parity used in the DRAG paper:

$$\mathcal E^x=\epsilon\mathcal E_\pi+\epsilon^3\mathcal E_x^{(3)}+\epsilon^5\mathcal E_x^{(5)},$$
$$\mathcal E^y=\epsilon^2\mathcal E_y^{(2)}+\epsilon^4\mathcal E_y^{(4)},\qquad\delta_1=\epsilon^2\delta_1^{(2)}+\epsilon^4\delta_1^{(4)}.$$

The superscript denotes the perturbative order, not a time derivative.

In [ ]:
Epi = sp.Function(r'\mathcal{E}_\pi', real=True)(t)
Delta = sp.Function(r'\Delta', real=True)(t)
lam = sp.Function(r'\lambda', real=True)(t)

Ex3 = sp.Function(r'\mathcal{E}_x^{(3)}', real=True)(t)
Ex5 = sp.Function(r'\mathcal{E}_x^{(5)}', real=True)(t)
Ey2 = sp.Function(r'\mathcal{E}_y^{(2)}', real=True)(t)
Ey4 = sp.Function(r'\mathcal{E}_y^{(4)}', real=True)(t)
d12 = sp.Function(r'\delta_1^{(2)}', real=True)(t)
d14 = sp.Function(r'\delta_1^{(4)}', real=True)(t)

Ex = eps*Epi + eps**3*Ex3 + eps**5*Ex5
Ey = eps**2*Ey2 + eps**4*Ey4
delta1 = eps**2*d12 + eps**4*d14

def ketbra(j, k):
    M = sp.zeros(3)
    M[j, k] = 1
    return M

Pi0, Pi1, Pi2 = (ketbra(j, j) for j in range(3))

def X(j, k):
    return ketbra(j, k) + ketbra(k, j)

def Y(j, k):
    return -I*ketbra(j, k) + I*ketbra(k, j)

X01, X12, X02 = X(0, 1), X(1, 2), X(0, 2)
Y01, Y12, Y02 = Y(0, 1), Y(1, 2), Y(0, 2)

H0 = Delta*Pi2
H = (delta1*Pi1 + (Delta + 2*delta1)*Pi2
     + Ex/2*(X01 + lam*X12)
     + Ey/2*(Y01 + lam*Y12))

## 3. Truncated BCH algebra

Every multiplication and commutator is truncated immediately. This is much faster than constructing the full symbolic BCH expression and discarding high orders only at the end. The switch `DROP_PARAMETER_DERIVATIVES` distinguishes the paper reference calculation from the generalized calculation.

In [ ]:
DROP_PARAMETER_DERIVATIVES = True

def trunc(expr, order=MAX_ORDER):
    return sp.series(sp.expand(expr), eps, 0, order + 1).removeO()

def mtrunc(M, order=MAX_ORDER):
    return sp.Matrix(M).applyfunc(lambda z: trunc(z, order))

def comm(A, B, order=MAX_ORDER):
    return mtrunc(A*B - B*A, order)

def ordered_dt(expr):
    # Motzoi adiabatic counting: each time derivative contributes one epsilon.
    out = eps*sp.diff(expr, t)
    if DROP_PARAMETER_DERIVATIVES:
        rules = {}
        for n in range(1, MAX_ORDER + 2):
            rules[sp.diff(Delta, t, n)] = 0
            rules[sp.diff(lam, t, n)] = 0
        out = out.xreplace(rules)
    return trunc(out)

def matrix_dt(M):
    return sp.Matrix(M).applyfunc(ordered_dt)

def transformed_hamiltonian(H_in, S, order=MAX_ORDER):
    result = mtrunc(H_in, order)
    nested = mtrunc(H_in, order)
    for n in range(1, order + 1):
        nested = comm(nested, S, order)
        result = mtrunc(result + nested/sp.factorial(n), order)

    nested = matrix_dt(S)
    for n in range(0, order):
        result = mtrunc(result - I*nested/sp.factorial(n + 1), order)
        nested = comm(nested, S, order)
    return mtrunc(result, order)

def at_order(expr, n):
    return sp.expand(trunc(expr, n)).coeff(eps, n)

def matrix_at_order(M, n):
    return sp.Matrix(M).applyfunc(lambda z: at_order(z, n))

In [ ]:
def clean(expr):
    expr = sp.cancel(sp.together(expr))
    num, den = sp.fraction(expr)
    return sp.cancel(sp.factor_terms(num)/sp.factor(den))

def coefficients(M, remove_ground=True):
    M = sp.Matrix(M)
    if remove_ground:
        M = M - M[0, 0]*sp.eye(3)
    return {
        'Pi1': M[1, 1], 'Pi2': M[2, 2],
        'x01': (M[0, 1] + M[1, 0])/2,
        'y01': (M[1, 0] - M[0, 1])/(2*I),
        'x12': (M[1, 2] + M[2, 1])/2,
        'y12': (M[2, 1] - M[1, 2])/(2*I),
        'x02': (M[0, 2] + M[2, 0])/2,
        'y02': (M[2, 0] - M[0, 2])/(2*I),
    }

def substitute_function(expr, variable, value, max_derivative=6):
    out = expr
    for n in range(max_derivative, 0, -1):
        out = out.subs(sp.diff(variable, t, n), sp.diff(value, t, n))
    return out.subs(variable, value).doit()

def apply_solutions(expr, solutions):
    out = expr
    for variable, value in solutions.items():
        if isinstance(out, sp.MatrixBase):
            out = out.applyfunc(lambda z: substitute_function(z, variable, value))
        else:
            out = substitute_function(out, variable, value)
    return out

def solve_linear_unique(equations, unknowns):
    equations = [clean(eq) for eq in equations]
    ans = sp.solve(equations, unknowns, dict=True, simplify=False)
    if len(ans) != 1:
        message = (f'Expected one solution, obtained {len(ans)}. ' +
                   'This signals an unfixed SW gauge or an insufficient generator basis.')
        raise ValueError(message)
    missing = [u for u in unknowns if u not in ans[0]]
    if missing:
        raise ValueError(f'Underdetermined generator/control gauge: {missing}')
    return {u: clean(ans[0][u]) for u in unknowns}

## 4. Homological equation for the next generator term

Let $R^{(n)}$ denote the order-$n$ residual after all lower-order controls and generator terms have been inserted. For a selected off-block matrix element $p\leftrightarrow q$, the next generator is determined by

$$R_{pq}^{(n)}+[H_0,S^{(n)}]_{pq}=0,$$

hence

$$S_{pq}^{(n)}=-\frac{R_{pq}^{(n)}}{E_p-E_q}.$$

This is the finite-dimensional Schrieffer--Wolff homological equation. We choose the minimal off-block gauge: all newly generated $P$--$P$ and $Q$--$Q$ entries are zero. The only explicit in-block gauge term is the original DRAG contribution $i\mathcal E_\pi Y_{01}/(2\Delta)$ at first order.

`leave_uncorrected` can be used to retain a selected residual, for example `{'y12'}`. Such a term remains in the audit table and is not silently discarded.

In [ ]:
CHANNEL_MATRIX = {
    'x12': X12, 'y12': Y12,
    'x02': X02, 'y02': Y02,
}

def selected_leakage_matrix(R, leave_uncorrected=frozenset()):
    c = coefficients(R, remove_ground=False)
    selected = sp.zeros(3)
    for name, op in CHANNEL_MATRIX.items():
        if name not in leave_uncorrected:
            selected += c[name]*op
    return selected

def homological_generator(R, leave_uncorrected=frozenset()):
    Rsel = selected_leakage_matrix(R, leave_uncorrected)
    energies = (sp.Integer(0), sp.Integer(0), Delta)
    G = sp.zeros(3)
    for p, q in ((0, 2), (1, 2)):
        G[p, q] = clean(-Rsel[p, q]/(energies[p] - energies[q]))
        G[q, p] = clean(-Rsel[q, p]/(energies[q] - energies[p]))
    # Anti-Hermiticity check under the declared real assumptions.
    assert sp.simplify(G + G.conjugate().T) == sp.zeros(3)
    return G

## 5. Recursive derivation

At each order the algorithm performs the following operations:

1. calculate $H^W$ with the accumulated single generator $S_{<n}$;
2. solve the computational-block conditions $c_{Y_{01}}=0$, $c_{\Pi_1}=0$, and $c_{X_{01}}=\mathcal E_\pi/2$ order by order;
3. insert the control solutions, extract the remaining leakage residual $R_{PQ}^{(n)}$;
4. solve the homological equation for $S^{(n)}$ and add $\epsilon^nS^{(n)}$ to the same generator;
5. recompute rather than treating the transformations as a product of unrelated frames.

At first order the homological equation determines the $Y_{12}$ part. The DRAG gauge then adds the $Y_{01}$ part. At second order the induced $X_{02}$ residual determines the familiar $Y_{02}$ generator term automatically.

In [ ]:
CONTROL_UNKNOWNS = {
    1: [],
    2: [Ey2, d12],
    3: [Ex3],
    4: [Ey4, d14],
    5: [Ex5],
}

def target_equations(HV, order):
    c = coefficients(HV)
    if order == 1:
        # Ex already contains eps*Epi. No variable is solved here.
        return []
    if order in (2, 4):
        return [at_order(c['y01'], order), at_order(c['Pi1'], order)]
    if order in (3, 5):
        # Higher X01 terms are removed so the dressed coefficient remains Epi/2.
        return [at_order(c['x01'], order)]
    return []

def derive_drag(drop_parameter_derivatives=True,
                leave_uncorrected=frozenset(),
                verbose=True):
    global DROP_PARAMETER_DERIVATIVES
    DROP_PARAMETER_DERIVATIVES = drop_parameter_derivatives

    S_total = sp.zeros(3)
    solutions = {}
    generator_orders = {}
    audit = {}

    for order in range(1, MAX_ORDER + 1):
        HV = transformed_hamiltonian(H, S_total)
        HV = apply_solutions(HV, solutions)

        unknowns = CONTROL_UNKNOWNS[order]
        if unknowns:
            new_solutions = solve_linear_unique(target_equations(HV, order), unknowns)
            solutions.update(new_solutions)

        HV = apply_solutions(transformed_hamiltonian(H, S_total), solutions)
        Rn = matrix_at_order(HV, order)
        audit[order] = {k: clean(v) for k, v in coefficients(Rn).items()}

        Gn = homological_generator(Rn, leave_uncorrected)
        if order == 1:
            # Motzoi gauge: canonical leakage generator plus the in-qubit term.
            Gn += I*Epi/(2*Delta)*Y01

        Gn = Gn.applyfunc(clean)
        generator_orders[order] = Gn
        S_total = mtrunc(S_total + eps**order*Gn)

        if verbose:
            active = [k for k in CHANNEL_MATRIX if audit[order][k] != 0]
            print(f'order {order}: solved {unknowns}; leakage before S^({order}) = {active}')

    HV_final = apply_solutions(transformed_hamiltonian(H, S_total), solutions)
    controls = {
        'Ex': trunc(apply_solutions(Ex, solutions)),
        'Ey': trunc(apply_solutions(Ey, solutions)),
        'delta1': trunc(apply_solutions(delta1, solutions)),
    }
    return {
        'S': S_total, 'S_orders': generator_orders,
        'solutions': solutions, 'controls': controls,
        'H_transformed': HV_final, 'audit_before_generator': audit,
        'leave_uncorrected': frozenset(leave_uncorrected),
    }

## 6. Constant-parameter reference run

This run is mandatory. It must reproduce the paper before the generalized expressions are trusted. In particular, inspect $S^{(1)}$ and $S^{(2)}$: in the present $W=V^\dagger$ convention they must have signs opposite to the generator written for $V$ in the 2009 paper.

In [ ]:
reference = derive_drag(drop_parameter_derivatives=True)

display(Markdown('### First two automatically obtained generator orders'))
display(Math(r'S^{(1)}=' + sp.latex(reference['S_orders'][1])))
display(Math(r'S^{(2)}=' + sp.latex(reference['S_orders'][2])))

In [ ]:
# Exact regression against Eq. (10) of Motzoi et al.
D, l = sp.symbols('Delta lambda', real=True, nonzero=True)

def constant_parameter_limit(expr):
    rules = {Delta: D, lam: l}
    for n in range(1, MAX_ORDER + 2):
        rules[sp.diff(Delta, t, n)] = 0
        rules[sp.diff(lam, t, n)] = 0
    return clean(expr.xreplace(rules).subs(eps, 1))

Ex_paper = (Epi + (l**2 - 4)*Epi**3/(8*D**2)
            - (13*l**4 - 76*l**2 + 112)*Epi**5/(128*D**4))
Ey_paper = (-sp.diff(Epi, t)/D
            + 33*(l**2 - 2)*Epi**2*sp.diff(Epi, t)/(24*D**3))
d1_paper = ((l**2 - 4)*Epi**2/(4*D)
            - (l**4 - 7*l**2 + 12)*Epi**4/(16*D**3))

derived_reference = {
    name: constant_parameter_limit(expr)
    for name, expr in reference['controls'].items()
}
expected_reference = {'Ex': Ex_paper, 'Ey': Ey_paper, 'delta1': d1_paper}

for name in ('Ex', 'Ey', 'delta1'):
    difference = clean(derived_reference[name] - expected_reference[name])
    display(Math(r'\Delta_{\mathrm{check},' + name + '}=' + sp.latex(difference)))
    assert difference == 0, (
        f'{name} does not reproduce Eq. (10). Do not run the generalized stage. '
        'The chosen higher-order SW gauge is not the paper gauge.'
    )

print('PASS: all three constant-parameter controls reproduce Eq. (10).')

## 7. Generalized run with $\Delta(t)$ and $\lambda(t)$

Run this cell only after the reference assertion passes. `leave_uncorrected` implements a partial SW transformation. For example, `{'y12'}` leaves the selected $Y_{12}$ residual in the transformed Hamiltonian. The term is still reported by the residual audit and therefore cannot be mistaken for a canceled contribution.

The pulse envelope itself remains arbitrary: only differentiability and the endpoint condition $\mathcal E_\pi(0)=\mathcal E_\pi(t_g)=0$ enter the derivation. A Gaussian or another specific envelope is needed only when the symbolic controls are evaluated or simulated.

In [ ]:
# Set to {'y12'} if this is precisely the lambda-dot channel that is to remain.
LEAVE_UNCORRECTED = frozenset()

generalized = derive_drag(
    drop_parameter_derivatives=False,
    leave_uncorrected=LEAVE_UNCORRECTED,
)

for name, lhs in [('Ex', r'\mathcal E^x'),
                  ('Ey', r'\mathcal E^y'),
                  ('delta1', r'\delta_1')]:
    display(Math(lhs + '(t)=' + sp.latex(clean(generalized['controls'][name].subs(eps, 1)))))

## 8. Residual audit

The following cell checks the *final* transformed Hamiltonian, not the residual before the corresponding $S^{(n)}$ was added. All selected leakage channels and all computational-block error conditions must vanish through the requested order. Channels listed in `LEAVE_UNCORRECTED` are displayed but not asserted to be zero.

In [ ]:
def final_audit(result):
    c = coefficients(result['H_transformed'])
    rows = []
    for order in range(1, MAX_ORDER + 1):
        for name in ('x01', 'y01', 'Pi1', 'x12', 'y12', 'x02', 'y02'):
            value = clean(at_order(c[name], order))
            if value != 0:
                rows.append((order, name, value))
    return rows

residual_rows = final_audit(generalized)
for order, name, value in residual_rows:
    display(Math(r'R_{' + name + '}^{(' + str(order) + ')}=' + sp.latex(value)))

for order, name, value in residual_rows:
    if name in CHANNEL_MATRIX and name not in generalized['leave_uncorrected']:
        raise AssertionError(f'Uncanceled selected leakage: order={order}, channel={name}')

## Interpretation of a failed reference test

Equation (10) of the 2009 paper does not print the complete $S^{(3)},S^{(4)},S^{(5)}$. Therefore Eq. (9) of the review plus the two displayed generator terms do not by themselves fix the higher-order SW gauge. If the reference assertion fails, the result is not a SymPy simplification issue: the minimal off-block gauge used here differs from the unpublished gauge used for Eq. (10). In that case one must enlarge `S^{(n)}` by explicit block-diagonal gauge components and determine them from additional conditions; one must **not** force the controls to the published values after the fact and call that a derivation.

The notebook deliberately stops before producing time-dependent formulas in that situation. This separates a verified generalization from an arbitrary gauge-dependent expression.